In [0]:
# ============================================================
# NOTEBOOK 0a — DIM GENERATOR
# O&G Rig Operations Intelligence Platform
# Builds all 15 bronze dimensions
# Depends on: Notebook_0b (staging must exist and have data)
# dim_date is static — run once manually, skip in daily job
# ============================================================

# CELL 1 — SETUP
import requests
from datetime import date, timedelta, datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, lit, row_number
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    LongType, DateType, BooleanType, TimestampType
)

API_BASE = "https://og-rig-api.onrender.com"
TODAY    = date.today()
NOW      = datetime.now()   # used for unknown member audit columns

# Create bronze schema
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

print(f"✓ Setup complete")
print(f"✓ API base : {API_BASE}")
print(f"✓ TODAY    : {TODAY}")

✓ Setup complete
✓ API base : https://og-rig-api.onrender.com
✓ TODAY    : 2026-05-31


In [0]:
# CELL 2 — DIM_DATE
# Static calendar table — run once, never rebuilt unless range changes
# Range : 2023-01-01 to 2035-12-31
# No SCD Type 2 — dates never change
# No -1 unknown member — every fact always has a valid date
# Grain : one row per calendar day

print("Building dim_date...")

spark.sql("DROP TABLE IF EXISTS workspace.bronze.dim_date")

start_date     = date(2023, 1, 1)
end_date       = date(2035, 12, 31)
day_suffix_map = {1:"st",2:"nd",3:"rd",21:"st",22:"nd",23:"rd",31:"st"}

date_rows = []
current   = start_date

while current <= end_date:
    d = current
    quarter   = (d.month - 1) // 3 + 1
    first_doq = date(d.year, 3 * quarter - 2, 1)
    last_doq  = (date(d.year, 3 * quarter + 1, 1) - timedelta(days=1)
                 if quarter < 4 else date(d.year, 12, 31))
    first_dom = date(d.year, d.month, 1)
    last_dom  = (date(d.year, d.month + 1, 1) - timedelta(days=1)
                 if d.month < 12 else date(d.year, 12, 31))
    iso_year, iso_week, _ = d.isocalendar()
    first_dow = d - timedelta(days=d.weekday())
    last_dow  = first_dow + timedelta(days=6)
    first_doy = date(d.year, 1, 1)
    last_doy  = date(d.year, 12, 31)
    epoch     = int((datetime(d.year, d.month, d.day) -
                     datetime(1970, 1, 1)).total_seconds())
    date_id   = int(d.strftime("%Y%m%d"))

    date_rows.append((
        date_id,                              # date_id_sk
        date_id,                              # date_id
        d,                                    # full_date
        epoch,                                # epoch
        d.strftime("%A"),                     # day_name
        d.strftime("%a"),                     # day_name_abbreviated
        d.weekday() + 1,                      # day_of_week (1=Mon)
        d.day,                                # day_of_month
        (d - first_doq).days + 1,             # day_of_quarter
        (d - first_doy).days + 1,             # day_of_year
        day_suffix_map.get(d.day, "th"),      # day_suffix
        -(-d.day // 7),                       # week_of_month (ceiling division)
        iso_week,                             # week_of_year
        f"{iso_year}-W{iso_week:02d}",        # week_of_year_iso
        first_dow,                            # first_day_of_week
        last_dow,                             # last_day_of_week
        d.weekday() >= 5,                     # is_weekend
        d.month,                              # month_actual
        d.strftime("%B"),                     # month_name
        d.strftime("%b"),                     # month_name_abbreviated
        d.strftime("%m%Y"),                   # mmyyyy
        d.strftime("%m%d%Y"),                 # mmddyyyy
        first_dom,                            # first_day_of_month
        last_dom,                             # last_day_of_month
        quarter,                              # quarter_actual
        f"Q{quarter}",                        # quarter_name
        first_doq,                            # first_day_of_quarter
        last_doq,                             # last_day_of_quarter
        d.year,                               # year_actual
        first_doy,                            # first_day_of_year
        last_doy,                             # last_day_of_year
    ))
    current += timedelta(days=1)

date_schema = StructType([
    StructField("date_id_sk",            IntegerType(), False),
    StructField("date_id",               IntegerType(), False),
    StructField("full_date",             DateType(),    False),
    StructField("epoch",                 LongType(),    False),
    StructField("day_name",              StringType(),  True),
    StructField("day_name_abbreviated",  StringType(),  True),
    StructField("day_of_week",           IntegerType(), True),
    StructField("day_of_month",          IntegerType(), True),
    StructField("day_of_quarter",        IntegerType(), True),
    StructField("day_of_year",           IntegerType(), True),
    StructField("day_suffix",            StringType(),  True),
    StructField("week_of_month",         IntegerType(), True),
    StructField("week_of_year",          IntegerType(), True),
    StructField("week_of_year_iso",      StringType(),  True),
    StructField("first_day_of_week",     DateType(),    True),
    StructField("last_day_of_week",      DateType(),    True),
    StructField("is_weekend",            BooleanType(), True),
    StructField("month_actual",          IntegerType(), True),
    StructField("month_name",            StringType(),  True),
    StructField("month_name_abbreviated",StringType(),  True),
    StructField("mmyyyy",                StringType(),  True),
    StructField("mmddyyyy",              StringType(),  True),
    StructField("first_day_of_month",    DateType(),    True),
    StructField("last_day_of_month",     DateType(),    True),
    StructField("quarter_actual",        IntegerType(), True),
    StructField("quarter_name",          StringType(),  True),
    StructField("first_day_of_quarter",  DateType(),    True),
    StructField("last_day_of_quarter",   DateType(),    True),
    StructField("year_actual",           IntegerType(), True),
    StructField("first_day_of_year",     DateType(),    True),
    StructField("last_day_of_year",      DateType(),    True),
])

df_date = (spark.createDataFrame(date_rows, schema=date_schema)
           .withColumn("loaded_at", current_timestamp()))

(df_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.dim_date"))

count = spark.sql("SELECT COUNT(*) as n FROM workspace.bronze.dim_date").collect()[0]["n"]
print(f"✓ dim_date: {count:,} rows ({start_date} to {end_date})")

# Spot check
spark.sql("""
    SELECT date_id_sk, date_id, full_date, day_name,
           month_name, quarter_name, year_actual, is_weekend
    FROM workspace.bronze.dim_date
    WHERE full_date IN ('2023-01-01','2024-07-04','2035-12-31')
    ORDER BY full_date
""").show()

Building dim_date...
✓ dim_date: 4,748 rows (2023-01-01 to 2035-12-31)
+----------+--------+----------+--------+----------+------------+-----------+----------+
|date_id_sk| date_id| full_date|day_name|month_name|quarter_name|year_actual|is_weekend|
+----------+--------+----------+--------+----------+------------+-----------+----------+
|  20230101|20230101|2023-01-01|  Sunday|   January|          Q1|       2023|      true|
|  20240704|20240704|2024-07-04|Thursday|      July|          Q3|       2024|     false|
|  20351231|20351231|2035-12-31|  Monday|  December|          Q4|       2035|     false|
+----------+--------+----------+--------+----------+------------+-----------+----------+



In [0]:
# CELL 3 — DIMS FROM API MASTER DATA
# dim_region, dim_rig, dim_crew_member
# Each dim gets a -1 unknown member row
# SCD Type 2: valid_from, valid_to, is_current
# Audit: loaded_at, updated_at
# SK generated using row_number() — deterministic, sequential, safe after joins

def fetch_master(endpoint):
    """Fetch all records from a master endpoint. Raises on error or empty data."""
    url  = f"{API_BASE}/{endpoint}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()["data"]
    if not data:
        raise Exception(f"API returned 0 records for {endpoint} — aborting")
    print(f"  ✓ {endpoint}: {len(data)} records fetched")
    return data


# ── dim_region ────────────────────────────────────────────────────────────────
print("Building dim_region...")
spark.sql("DROP TABLE IF EXISTS workspace.bronze.dim_region")

regions       = fetch_master("master/regions")
df_region_raw = spark.createDataFrame(regions)

# row_number() over region_name gives clean sequential SKs
w_region = Window.orderBy("region_name")
df_region_data = (df_region_raw
    .withColumn("region_id_sk", row_number().over(w_region))
    .withColumn("region_id",    F.col("region_id_sk"))   # no business ID in API — copy sk
    .withColumn("valid_from",   lit(TODAY))
    .withColumn("valid_to",     lit(None).cast("date"))
    .withColumn("is_current",   lit(True))
    .withColumn("loaded_at",    current_timestamp())
    .withColumn("updated_at",   current_timestamp())
    .select("region_id_sk", "region_id", "region_name",
            "basin", "state",
            "valid_from", "valid_to", "is_current",
            "loaded_at", "updated_at"))

# Unknown member row (-1)
unknown_region = spark.createDataFrame([(
    -1, -1, "Unknown", "Unknown", "Unknown",
    TODAY, None, True, NOW, NOW
)], schema=df_region_data.schema)

df_region = unknown_region.union(df_region_data)

(df_region.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.dim_region"))

count = spark.sql("SELECT COUNT(*) as n FROM workspace.bronze.dim_region").collect()[0]["n"]
print(f"✓ dim_region: {count} rows (incl. -1 unknown member)")


# ── dim_rig ───────────────────────────────────────────────────────────────────
print("Building dim_rig...")
spark.sql("DROP TABLE IF EXISTS workspace.bronze.dim_rig")

rigs       = fetch_master("master/rigs")
df_rig_raw = spark.createDataFrame(rigs)

# API field is 'region' — rename to 'region_name' before joining
# Join to df_region excluding the -1 unknown row to get region_id_sk
w_rig = Window.orderBy("rig_id")
df_rig_data = (df_rig_raw
    .withColumnRenamed("region", "region_name")
    .join(
        df_region.filter(F.col("region_id_sk") != -1)
                 .select("region_id_sk", "region_id", "region_name"),
        on="region_name", how="left"
    )
    # Fill -1 for any rigs whose region didn't match
    .withColumn("region_id_sk", F.coalesce(F.col("region_id_sk"), lit(-1)))
    .withColumn("region_id",    F.coalesce(F.col("region_id"),    lit(-1)))
    # row_number after join — deterministic sequential SK
    .withColumn("rig_id_sk",    row_number().over(w_rig))
    # rig_id comes from the API — already 'RIG-001' etc.
    .withColumn("valid_from",   lit(TODAY))
    .withColumn("valid_to",     lit(None).cast("date"))
    .withColumn("is_current",   lit(True))
    .withColumn("loaded_at",    current_timestamp())
    .withColumn("updated_at",   current_timestamp())
    .select("rig_id_sk", "rig_id", "rig_name",
            "region_id_sk", "region_id", "region_name",
            "rig_type", "commission_year",
            "valid_from", "valid_to", "is_current",
            "loaded_at", "updated_at"))

# Unknown member row (-1)
unknown_rig = spark.createDataFrame([(
    -1, "Unknown", "Unknown",
    -1, -1, "Unknown",
    "Unknown", 0,
    TODAY, None, True, NOW, NOW
)], schema=df_rig_data.schema)

df_rig = unknown_rig.union(df_rig_data)

(df_rig.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.dim_rig"))

count = spark.sql("SELECT COUNT(*) as n FROM workspace.bronze.dim_rig").collect()[0]["n"]
print(f"✓ dim_rig: {count} rows (incl. -1 unknown member)")


# ── dim_crew_member ───────────────────────────────────────────────────────────
print("Building dim_crew_member...")
spark.sql("DROP TABLE IF EXISTS workspace.bronze.dim_crew_member")

crew        = fetch_master("master/crew")
df_crew_raw = spark.createDataFrame(crew)

# API field is 'region' — rename before joining
w_crew = Window.orderBy("crew_id")
df_crew_data = (df_crew_raw
    .withColumnRenamed("region", "region_name")
    .join(
        df_region.filter(F.col("region_id_sk") != -1)
                 .select("region_id_sk", "region_id", "region_name"),
        on="region_name", how="left"
    )
    # Fill -1 for any crew whose region didn't match
    .withColumn("region_id_sk", F.coalesce(F.col("region_id_sk"), lit(-1)))
    .withColumn("region_id",    F.coalesce(F.col("region_id"),    lit(-1)))
    # row_number after join — deterministic sequential SK
    .withColumn("crew_id_sk",   row_number().over(w_crew))
    # crew_id comes from the API — already 'CREW-001' etc.
    .withColumn("valid_from",   lit(TODAY))
    .withColumn("valid_to",     lit(None).cast("date"))
    .withColumn("is_current",   lit(True))
    .withColumn("loaded_at",    current_timestamp())
    .withColumn("updated_at",   current_timestamp())
    .select("crew_id_sk", "crew_id",
            "role", "certification_level", "years_experience",
            "region_id_sk", "region_id", "region_name",
            "valid_from", "valid_to", "is_current",
            "loaded_at", "updated_at"))

# Unknown member row (-1)
unknown_crew = spark.createDataFrame([(
    -1, "Unknown",
    "Unknown", "Unknown", 0,
    -1, -1, "Unknown",
    TODAY, None, True, NOW, NOW
)], schema=df_crew_data.schema)

df_crew = unknown_crew.union(df_crew_data)

(df_crew.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.dim_crew_member"))

count = spark.sql("SELECT COUNT(*) as n FROM workspace.bronze.dim_crew_member").collect()[0]["n"]
print(f"✓ dim_crew_member: {count} rows (incl. -1 unknown member)")

print("\n✓ API master dims complete")

Building dim_region...
  ✓ master/regions: 5 records fetched


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ dim_region: 6 rows (incl. -1 unknown member)
Building dim_rig...
  ✓ master/rigs: 20 records fetched
✓ dim_rig: 21 rows (incl. -1 unknown member)
Building dim_crew_member...
  ✓ master/crew: 50 records fetched
✓ dim_crew_member: 51 rows (incl. -1 unknown member)

✓ API master dims complete


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# CELL 4 — DIMS DISCOVERED FROM STAGING
# Builds dims from distinct values in staging tables
# Each gets: [entity]_id_sk (generated via row_number),
# natural key (from data), SCD Type 2 columns,
# -1 unknown member, loaded_at, updated_at

def build_staging_dim(table_name, sql, sk_col, order_col, extra_cols, select_cols):
    """
    Generic helper to build a staging-discovered dim.
    - sql        : query returning the natural key column
    - sk_col     : name of the surrogate key column e.g. 'equipment_id_sk'
    - order_col  : column to order by for row_number (usually the natural key)
    - extra_cols : dict of {col_name: literal_value} for hardcoded attributes
    - select_cols: final column order
    """
    df = spark.sql(sql)
    if df.count() == 0:
        raise Exception(f"Staging query for {table_name} returned 0 rows — aborting")

    w = Window.orderBy(order_col)
    df = df.withColumn(sk_col, row_number().over(w))

    for col_name, col_val in extra_cols.items():
        df = df.withColumn(col_name, lit(col_val))

    df = (df
        .withColumn("valid_from", lit(TODAY))
        .withColumn("valid_to",   lit(None).cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn("loaded_at",  current_timestamp())
        .withColumn("updated_at", current_timestamp())
        .select(select_cols))

    return df


def add_unknown_and_save(df, table_name, unknown_row_dict):
    """
    Prepends a -1 unknown member row and saves to Delta.
    unknown_row_dict values must match df.schema column order exactly.
    """
    spark.sql(f"DROP TABLE IF EXISTS workspace.bronze.{table_name}")
    unknown_df = spark.createDataFrame(
        [tuple(unknown_row_dict.values())], schema=df.schema
    )
    final_df = unknown_df.union(df)
    (final_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"workspace.bronze.{table_name}"))
    count = spark.sql(
        f"SELECT COUNT(*) as n FROM workspace.bronze.{table_name}"
    ).collect()[0]["n"]
    print(f"✓ {table_name}: {count} rows (incl. -1 unknown member)")


# ── dim_equipment ─────────────────────────────────────────────────────────────
print("Building dim_equipment...")
df_equipment = build_staging_dim(
    table_name  = "dim_equipment",
    sql         = "SELECT DISTINCT equipment_type FROM workspace.staging.stg_equipment_events",
    sk_col      = "equipment_id_sk",
    order_col   = "equipment_type",
    extra_cols  = {
        "criticality"            : "Critical",
        "expected_lifespan_days" : 1825,
    },
    select_cols = ["equipment_id_sk", "equipment_type",
                   "criticality", "expected_lifespan_days",
                   "valid_from", "valid_to", "is_current",
                   "loaded_at", "updated_at"]
)
add_unknown_and_save(df_equipment, "dim_equipment", {
    "equipment_id_sk"        : -1,
    "equipment_type"         : "Unknown",
    "criticality"            : "Unknown",
    "expected_lifespan_days" : 0,
    "valid_from"             : TODAY,
    "valid_to"               : None,
    "is_current"             : True,
    "loaded_at"              : NOW,
    "updated_at"             : NOW,
})


# ── dim_status ────────────────────────────────────────────────────────────────
print("Building dim_status...")
df_status = build_staging_dim(
    table_name  = "dim_status",
    sql         = """
        SELECT DISTINCT status AS status_name FROM workspace.staging.stg_rig_readings
        UNION
        SELECT DISTINCT incident_status AS status_name FROM workspace.staging.stg_incidents
    """,
    sk_col      = "status_id_sk",
    order_col   = "status_name",
    extra_cols  = {"status_category": "Operational"},
    select_cols = ["status_id_sk", "status_name", "status_category",
                   "valid_from", "valid_to", "is_current",
                   "loaded_at", "updated_at"]
)
add_unknown_and_save(df_status, "dim_status", {
    "status_id_sk"    : -1,
    "status_name"     : "Unknown",
    "status_category" : "Unknown",
    "valid_from"      : TODAY,
    "valid_to"        : None,
    "is_current"      : True,
    "loaded_at"       : NOW,
    "updated_at"      : NOW,
})


# ── dim_failure_type ──────────────────────────────────────────────────────────
print("Building dim_failure_type...")
df_failure = build_staging_dim(
    table_name  = "dim_failure_type",
    sql         = """
        SELECT DISTINCT failure_type
        FROM workspace.staging.stg_equipment_events
        WHERE failure_type IS NOT NULL
    """,
    sk_col      = "failure_type_id_sk",
    order_col   = "failure_type",
    extra_cols  = {
        "failure_category" : "Mechanical",
        "severity"         : "High",
    },
    select_cols = ["failure_type_id_sk", "failure_type",
                   "failure_category", "severity",
                   "valid_from", "valid_to", "is_current",
                   "loaded_at", "updated_at"]
)
add_unknown_and_save(df_failure, "dim_failure_type", {
    "failure_type_id_sk" : -1,
    "failure_type"       : "Unknown",
    "failure_category"   : "Unknown",
    "severity"           : "Unknown",
    "valid_from"         : TODAY,
    "valid_to"           : None,
    "is_current"         : True,
    "loaded_at"          : NOW,
    "updated_at"         : NOW,
})


# ── dim_maintenance_type ──────────────────────────────────────────────────────
print("Building dim_maintenance_type...")
df_maint_type = build_staging_dim(
    table_name  = "dim_maintenance_type",
    sql         = "SELECT DISTINCT maintenance_type FROM workspace.staging.stg_maintenance",
    sk_col      = "maintenance_type_id_sk",
    order_col   = "maintenance_type",
    extra_cols  = {
        "schedule_type" : "Scheduled",
        "is_planned"    : True,
    },
    select_cols = ["maintenance_type_id_sk", "maintenance_type",
                   "schedule_type", "is_planned",
                   "valid_from", "valid_to", "is_current",
                   "loaded_at", "updated_at"]
)
add_unknown_and_save(df_maint_type, "dim_maintenance_type", {
    "maintenance_type_id_sk" : -1,
    "maintenance_type"       : "Unknown",
    "schedule_type"          : "Unknown",
    "is_planned"             : False,
    "valid_from"             : TODAY,
    "valid_to"               : None,
    "is_current"             : True,
    "loaded_at"              : NOW,
    "updated_at"             : NOW,
})


# ── dim_contractor ────────────────────────────────────────────────────────────
print("Building dim_contractor...")
df_contractor = build_staging_dim(
    table_name  = "dim_contractor",
    sql         = "SELECT DISTINCT contractor_name FROM workspace.staging.stg_maintenance",
    sk_col      = "contractor_id_sk",
    order_col   = "contractor_name",
    extra_cols  = {
        "contractor_type" : "Oilfield Services",
        "country"         : "USA",
    },
    select_cols = ["contractor_id_sk", "contractor_name",
                   "contractor_type", "country",
                   "valid_from", "valid_to", "is_current",
                   "loaded_at", "updated_at"]
)
add_unknown_and_save(df_contractor, "dim_contractor", {
    "contractor_id_sk" : -1,
    "contractor_name"  : "Unknown",
    "contractor_type"  : "Unknown",
    "country"          : "Unknown",
    "valid_from"       : TODAY,
    "valid_to"         : None,
    "is_current"       : True,
    "loaded_at"        : NOW,
    "updated_at"       : NOW,
})


# ── dim_downtime_reason ───────────────────────────────────────────────────────
print("Building dim_downtime_reason...")
df_downtime = build_staging_dim(
    table_name  = "dim_downtime_reason",
    sql         = """
        SELECT DISTINCT downtime_reason
        FROM workspace.staging.stg_rig_readings
        WHERE downtime_reason IS NOT NULL
    """,
    sk_col      = "downtime_reason_id_sk",
    order_col   = "downtime_reason",
    extra_cols  = {
        "downtime_category" : "Controllable",
        "is_controllable"   : True,
    },
    select_cols = ["downtime_reason_id_sk", "downtime_reason",
                   "downtime_category", "is_controllable",
                   "valid_from", "valid_to", "is_current",
                   "loaded_at", "updated_at"]
)
add_unknown_and_save(df_downtime, "dim_downtime_reason", {
    "downtime_reason_id_sk" : -1,
    "downtime_reason"       : "Unknown",
    "downtime_category"     : "Unknown",
    "is_controllable"       : False,
    "valid_from"            : TODAY,
    "valid_to"              : None,
    "is_current"            : True,
    "loaded_at"             : NOW,
    "updated_at"            : NOW,
})


# ── dim_shift ─────────────────────────────────────────────────────────────────
# Built manually (not via helper) to add start_hour and end_hour
print("Building dim_shift...")
spark.sql("DROP TABLE IF EXISTS workspace.bronze.dim_shift")

shift_count = spark.sql(
    "SELECT COUNT(DISTINCT shift) as n FROM workspace.staging.stg_crew"
).collect()[0]["n"]
if shift_count == 0:
    raise Exception("dim_shift staging query returned 0 rows — aborting")

w_shift = Window.orderBy("shift_name")
df_shift_data = (spark.sql(
    "SELECT DISTINCT shift AS shift_name FROM workspace.staging.stg_crew"
)
    .withColumn("shift_id_sk", row_number().over(w_shift))
    .withColumn("start_hour",  F.when(F.col("shift_name") == "Day", 6).otherwise(18))
    .withColumn("end_hour",    F.when(F.col("shift_name") == "Day", 18).otherwise(6))
    .withColumn("valid_from",  lit(TODAY))
    .withColumn("valid_to",    lit(None).cast("date"))
    .withColumn("is_current",  lit(True))
    .withColumn("loaded_at",   current_timestamp())
    .withColumn("updated_at",  current_timestamp())
    .select("shift_id_sk", "shift_name",
            "start_hour", "end_hour",
            "valid_from", "valid_to", "is_current",
            "loaded_at", "updated_at"))

unknown_shift = spark.createDataFrame([(
    -1, "Unknown", -1, -1,
    TODAY, None, True, NOW, NOW
)], schema=df_shift_data.schema)

df_shift = unknown_shift.union(df_shift_data)

(df_shift.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.dim_shift"))

count = spark.sql("SELECT COUNT(*) as n FROM workspace.bronze.dim_shift").collect()[0]["n"]
print(f"✓ dim_shift: {count} rows (incl. -1 unknown member)")


# ── dim_well_type ─────────────────────────────────────────────────────────────
print("Building dim_well_type...")
df_well_type = build_staging_dim(
    table_name  = "dim_well_type",
    sql         = "SELECT DISTINCT well_type FROM workspace.staging.stg_rig_readings",
    sk_col      = "well_type_id_sk",
    order_col   = "well_type",
    extra_cols  = {"drilling_direction": "Unknown"},
    select_cols = ["well_type_id_sk", "well_type", "drilling_direction",
                   "valid_from", "valid_to", "is_current",
                   "loaded_at", "updated_at"]
)
add_unknown_and_save(df_well_type, "dim_well_type", {
    "well_type_id_sk"    : -1,
    "well_type"          : "Unknown",
    "drilling_direction" : "Unknown",
    "valid_from"         : TODAY,
    "valid_to"           : None,
    "is_current"         : True,
    "loaded_at"          : NOW,
    "updated_at"         : NOW,
})


# ── dim_cost_category ─────────────────────────────────────────────────────────
print("Building dim_cost_category...")
df_cost = build_staging_dim(
    table_name  = "dim_cost_category",
    sql         = "SELECT DISTINCT cost_category FROM workspace.staging.stg_rig_readings",
    sk_col      = "cost_category_id_sk",
    order_col   = "cost_category",
    extra_cols  = {"cost_type": "OPEX"},
    select_cols = ["cost_category_id_sk", "cost_category",
                   "cost_type",
                   "valid_from", "valid_to", "is_current",
                   "loaded_at", "updated_at"]
)
add_unknown_and_save(df_cost, "dim_cost_category", {
    "cost_category_id_sk" : -1,
    "cost_category"       : "Unknown",
    "cost_type"           : "Unknown",
    "valid_from"          : TODAY,
    "valid_to"            : None,
    "is_current"          : True,
    "loaded_at"           : NOW,
    "updated_at"          : NOW,
})


# ── dim_incident_type ─────────────────────────────────────────────────────────
print("Building dim_incident_type...")
df_incident_type = build_staging_dim(
    table_name  = "dim_incident_type",
    sql         = "SELECT DISTINCT incident_type FROM workspace.staging.stg_incidents",
    sk_col      = "incident_type_id_sk",
    order_col   = "incident_type",
    extra_cols  = {
        "incident_category" : "Safety",
        "severity_level"    : "Medium",
    },
    select_cols = ["incident_type_id_sk", "incident_type",
                   "incident_category", "severity_level",
                   "valid_from", "valid_to", "is_current",
                   "loaded_at", "updated_at"]
)
add_unknown_and_save(df_incident_type, "dim_incident_type", {
    "incident_type_id_sk" : -1,
    "incident_type"       : "Unknown",
    "incident_category"   : "Unknown",
    "severity_level"      : "Unknown",
    "valid_from"          : TODAY,
    "valid_to"            : None,
    "is_current"          : True,
    "loaded_at"           : NOW,
    "updated_at"          : NOW,
})


# ── dim_well ──────────────────────────────────────────────────────────────────
# Grain: well_name (unique across all rigs)
# well_type: MAX() used for deterministic result across runs
print("Building dim_well...")
spark.sql("DROP TABLE IF EXISTS workspace.bronze.dim_well")

w_well = Window.orderBy("well_name")
df_well_data = (spark.sql("""
    SELECT
        well_name,
        MAX(well_type) AS well_type
    FROM workspace.staging.stg_rig_readings
    GROUP BY well_name
""")
    .withColumn("well_id_sk",  row_number().over(w_well))
    .withColumn("valid_from",  lit(TODAY))
    .withColumn("valid_to",    lit(None).cast("date"))
    .withColumn("is_current",  lit(True))
    .withColumn("loaded_at",   current_timestamp())
    .withColumn("updated_at",  current_timestamp())
    .select("well_id_sk", "well_name", "well_type",
            "valid_from", "valid_to", "is_current",
            "loaded_at", "updated_at"))

if df_well_data.count() == 0:
    raise Exception("dim_well staging query returned 0 rows — aborting")

unknown_well = spark.createDataFrame([(
    -1, "Unknown", "Unknown",
    TODAY, None, True, NOW, NOW
)], schema=df_well_data.schema)

df_well = unknown_well.union(df_well_data)

(df_well.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.dim_well"))

count = spark.sql("SELECT COUNT(*) as n FROM workspace.bronze.dim_well").collect()[0]["n"]
print(f"✓ dim_well: {count} rows (incl. -1 unknown member)")

print("\n✓ All staging-discovered dims complete")

Building dim_equipment...


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ dim_equipment: 6 rows (incl. -1 unknown member)
Building dim_status...
✓ dim_status: 6 rows (incl. -1 unknown member)
Building dim_failure_type...
✓ dim_failure_type: 6 rows (incl. -1 unknown member)
Building dim_maintenance_type...
✓ dim_maintenance_type: 4 rows (incl. -1 unknown member)
Building dim_contractor...
✓ dim_contractor: 6 rows (incl. -1 unknown member)
Building dim_downtime_reason...
✓ dim_downtime_reason: 8 rows (incl. -1 unknown member)
Building dim_shift...
✓ dim_shift: 3 rows (incl. -1 unknown member)
Building dim_well_type...
✓ dim_well_type: 4 rows (incl. -1 unknown member)
Building dim_cost_category...
✓ dim_cost_category: 9 rows (incl. -1 unknown member)
Building dim_incident_type...
✓ dim_incident_type: 8 rows (incl. -1 unknown member)
Building dim_well...
✓ dim_well: 61 rows (incl. -1 unknown member)

✓ All staging-discovered dims complete


In [0]:
# CELL 5 — VERIFICATION
# Check all 15 dims: row counts, column counts, SCD columns, -1 unknown member

dims = [
    "dim_date",
    "dim_region", "dim_rig", "dim_crew_member",
    "dim_equipment", "dim_status", "dim_failure_type",
    "dim_maintenance_type", "dim_contractor", "dim_downtime_reason",
    "dim_shift", "dim_well_type", "dim_cost_category",
    "dim_incident_type", "dim_well",
]
no_scd  = {"dim_date"}   # date dim has no SCD Type 2 columns
no_neg1 = {"dim_date"}   # date dim has no -1 unknown member

print(f"{'Table':<35} {'Rows':>6}  {'Cols':>5}  {'SCD':>4}  {'-1':>4}  {'loaded_at':>10}")
print("-" * 75)
total  = 0
issues = []

for t in dims:
    df       = spark.table(f"workspace.bronze.{t}")
    rows     = df.count()
    cols     = len(df.columns)
    total   += rows
    col_list = df.columns

    has_scd    = (
        t in no_scd or
        all(c in col_list for c in ["valid_from", "valid_to", "is_current"])
    )
    has_loaded = "loaded_at" in col_list

    # Check -1 unknown member exists
    if t not in no_neg1:
        sk_col = [c for c in col_list if c.endswith("_id_sk")]
        if sk_col:
            neg1_count = df.filter(F.col(sk_col[0]) == -1).count()
            has_neg1   = neg1_count == 1
        else:
            has_neg1 = False
    else:
        has_neg1 = True

    if not has_scd:    issues.append(f"  ⚠ {t}: missing SCD columns")
    if not has_neg1:   issues.append(f"  ⚠ {t}: missing -1 unknown member")
    if not has_loaded: issues.append(f"  ⚠ {t}: missing loaded_at column")

    print(f"workspace.bronze.{t:<19} {rows:>6,}  {cols:>5}  "
          f"{'✓' if has_scd else '✗':>4}  "
          f"{'✓' if has_neg1 else '✗':>4}  "
          f"{'✓' if has_loaded else '✗':>10}")

print("-" * 75)
print(f"{'Total rows':<35} {total:>6,}")

if issues:
    print("\nISSUES FOUND:")
    for i in issues:
        print(i)
    raise Exception("Dim validation failed — review issues above before running Notebook_0c")
else:
    print("\n✓ All 15 dims validated — Notebook 0a complete")
    print("✓ Safe to proceed to Notebook_0c")

Table                                 Rows   Cols   SCD    -1   loaded_at
---------------------------------------------------------------------------
workspace.bronze.dim_date             4,748     32     ✓     ✓           ✓
workspace.bronze.dim_region               6     10     ✓     ✓           ✓
workspace.bronze.dim_rig                 21     13     ✓     ✓           ✓
workspace.bronze.dim_crew_member         51     13     ✓     ✓           ✓
workspace.bronze.dim_equipment            6      9     ✓     ✓           ✓
workspace.bronze.dim_status               6      8     ✓     ✓           ✓
workspace.bronze.dim_failure_type         6      9     ✓     ✓           ✓
workspace.bronze.dim_maintenance_type      4      9     ✓     ✓           ✓
workspace.bronze.dim_contractor           6      9     ✓     ✓           ✓
workspace.bronze.dim_downtime_reason      8      9     ✓     ✓           ✓
workspace.bronze.dim_shift                3      9     ✓     ✓           ✓
workspace.bronze.dim_wel